# KG-Commit: Online Knowledge Graph Simulation

Simulates the real deployment scenario: commits arrive one-by-one in chronological order.
For each commit the loop does exactly **four steps in order**:

```
1. extract_kg_features(G, commit)   ← query G BEFORE this commit is known
2. predict(features)                ← make a prediction
3. evaluate(prediction, true_label) ← record the result
4. update_kg(G, commit)             ← add this commit's knowledge to G
```

`G` at step 1 only knows about commits **strictly before** the current one — zero label leakage.

---

### What gets added to G per commit (by tier)

| Tier | Entities | Needs |
|------|----------|-------|
| Core | `COMMIT` `TIME` `INTERVAL` `LABEL` | local CSV |
| File | `AUTHOR` `FILE` `FILE_TYPE` `DIR` `EXTERNAL_PACKAGE` | diff_text (server) |
| Within-file | `CLASS` `FUNCTION` `FUNCTION_SIGNATURE` `VARIABLE` `DATA_TYPE` | diff_text (server) |
| Finer | `ISSUE` `BRANCH` | git repo (server) |

### KG features extracted before each commit

| Feature | Source in G |
|---------|-------------|
| `kg_project_commit_count` | count of COMMIT nodes so far |
| `kg_project_bug_rate` | fraction of past commits labelled buggy |
| `kg_author_commit_count` | AUTHOR node counter |
| `kg_author_bug_rate` | AUTHOR node counter |
| `kg_file_change_count` | sum of FILE node counters for touched files |
| `kg_file_bug_rate` | mean bug rate across touched files |
| `kg_file_unique_authors` | mean unique-author count across touched files |

## 0 — Imports & Configuration

## 0.1 — Install Dependencies

Install NetworkX, Plotly (for 3D visualizations), and GitPython (for branch indexing).
These are loaded into memory once at the start; if GitPython is unavailable the BRANCH tier
will be skipped gracefully.

In [ ]:
%pip install plotly

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install gitpython

## 0.2 — Initialize Imports & Configuration

Import all required libraries (NetworkX, Pandas, scikit-learn, etc.) and set up paths.
The notebook operates on the **apache_groovy** project; change `PROJECT_NAME` to run on a different project.
All data paths resolve relative to the repo root:
- **LOCAL_CSV**: 18 handcrafted features, no diff text (basic.csv)
- **MASTER_DIFF_CSV**: All projects with diff_text, rebuilt by build_diffs.py
- **Auxiliary data**: Issues, commits↔issue map, cloned git repos for BRANCH tier

In [ ]:
import networkx as nx
import pandas as pd
import re
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# ── Project under analysis ───────────────────────────────────────────
PROJECT_NAME = "apache_groovy"                # CSV / projects/ slug
REPO_SLUG    = PROJECT_NAME.removeprefix("apache_")   # repo folder name ('groovy')

# ── Paths (all local now) ────────────────────────────────────────────
DATA_DIR     = Path("../../data/apachejit")
PROJECTS_DIR = DATA_DIR / "projects"
REPOS_DIR    = Path("../../repos/apache")

MASTER_DIFF_CSV   = DATA_DIR / "apachejit_with_diffs_rebuilt.csv"   # built by build_diffs.py
LOCAL_CSV         = PROJECTS_DIR / f"{PROJECT_NAME}.csv"            # 18 features, no diff
DIFF_TEXT_CSV     = PROJECTS_DIR / f"{PROJECT_NAME}_diff.csv"       # split output, has diff_text
ISSUE_TEXTS_CSV   = DATA_DIR / "issue_texts.csv"                    # built by fetch_issues.py
COMMIT_ISSUE_MAP  = DATA_DIR / "commit_issue_map.csv"               # built by fetch_issues.py
REPO_PATH         = REPOS_DIR / REPO_SLUG                           # cloned git repo (Tier-5)

print(f"NetworkX        : {nx.__version__}")
print(f"Local CSV       : {LOCAL_CSV}        exists={LOCAL_CSV.exists()}")
print(f"Master diff     : {MASTER_DIFF_CSV}  exists={MASTER_DIFF_CSV.exists()}")
print(f"Diff CSV        : {DIFF_TEXT_CSV}    exists={DIFF_TEXT_CSV.exists()}")
print(f"Issue texts     : {ISSUE_TEXTS_CSV}  exists={ISSUE_TEXTS_CSV.exists()}")
print(f"Commit↔issue    : {COMMIT_ISSUE_MAP} exists={COMMIT_ISSUE_MAP.exists()}")
print(f"Repo            : {REPO_PATH}        exists={REPO_PATH.exists()}")

## 0.3 — Split Master Diff CSV into Per-Project Files

`apachejit_with_diffs_rebuilt.csv` (server-side) holds every commit **with** its `diff_text`.
Split it **once** into `data/apachejit/projects/<slug>_diff.csv` so the rest of the notebook
loads only the current project instead of the full multi-GB master file.

`<slug>` = `project` value with `/` replaced by `_` (e.g. `apache/groovy` → `apache_groovy`),
matching the existing no-diff per-project files in the same folder.

If the master file is absent (e.g. running locally) the split is skipped and the notebook
expects the per-project `*_diff.csv` to already exist.

In [ ]:
if MASTER_DIFF_CSV.exists():
    PROJECTS_DIR.mkdir(parents=True, exist_ok=True)
    _master = pd.read_csv(MASTER_DIFF_CSV)
    print(f"Master: {len(_master):,} rows, {_master.project.nunique()} projects\n")

    for proj, grp in _master.groupby("project"):
        slug = proj.replace("/", "_")
        dest = PROJECTS_DIR / f"{slug}_diff.csv"
        grp.to_csv(dest, index=False)
        print(f"  {proj:28} -> {dest.name:34} ({len(grp):,} rows)")

    del _master
    print(f"\nSplit complete. Current project file: {DIFF_TEXT_CSV.name}"
          f"  exists={DIFF_TEXT_CSV.exists()}")
else:
    print(f"Master diff CSV not found: {MASTER_DIFF_CSV}")
    print(f"Skipping split. Expecting per-project file: {DIFF_TEXT_CSV}")
    print(f"  exists={DIFF_TEXT_CSV.exists()}")

## 0.4 — Load Core Data

Load the CSV of commits with handcrafted features, sort by timestamp, and load optional auxiliary data:
- **diff_text**: Git patch information, parsed to extract files, authors, imports, classes, functions, variables, and issues
- **commit_issue_map**: Pre-extracted JIRA issue IDs per commit (from fetch_issues.py Phase 1)
- **issue_texts**: Fetched JIRA metadata (title, description, status) from JIRA API (Phase 2)

These are all optional—the KG builds successfully with just the basic CSV, though finer tiers (FILE, AUTHOR, ISSUE) will be less rich.

In [ ]:
# Sort by author_date — this is the commit stream order
df = pd.read_csv(LOCAL_CSV).sort_values("author_date").reset_index(drop=True)
print(f"Commits        : {len(df):,}  |  {df.author_date.min()} → {df.author_date.max()}")

# ── diff_text (per-project split from build_diffs.py) ─────────────
if DIFF_TEXT_CSV.exists():
    _diff_df = pd.read_csv(DIFF_TEXT_CSV)
    diff_map = dict(zip(_diff_df.commit_id, _diff_df.diff_text))
    print(f"diff_text      : {len(diff_map):,} commits loaded")
else:
    diff_map = {}
    print("diff_text      : not found — file / author / within-file tiers will be skipped")

# ── ISSUE references (from fetch_issues.py) ───────────────────────
commit_to_issues = defaultdict(list)
if COMMIT_ISSUE_MAP.exists():
    _m = pd.read_csv(COMMIT_ISSUE_MAP, dtype=str)
    for cid, iid in zip(_m.commit_id, _m.issue_id):
        commit_to_issues[cid].append(iid)
    print(f"commit→issue   : {len(commit_to_issues):,} commits linked to JIRA issues")
else:
    print("commit→issue   : not found — ISSUE tier will fall back to regex")

issue_meta = {}
if ISSUE_TEXTS_CSV.exists():
    _i = pd.read_csv(ISSUE_TEXTS_CSV, dtype=str, keep_default_na=False)
    for r in _i.itertuples(index=False):
        issue_meta[r.issue_id] = {
            "title":       getattr(r, "title", "")       or "",
            "description": getattr(r, "description", "") or "",
            "status":      getattr(r, "status", "")      or "",
            "created":     getattr(r, "created", "")     or "",
            "updated":     getattr(r, "updated", "")     or "",
        }
    print(f"issue texts    : {len(issue_meta):,} JIRA issues with metadata (title/description/status/…)")
else:
    print("issue texts    : not found — ISSUE nodes will lack title / description")

## 2 — Parser Functions

A collection of **pure functions** (no side effects on `G`) that extract structured information from raw diff text.
Each parser:
- Returns early if the input is not a string, defaulting to an empty collection
- Uses regex patterns tuned for both Python and Java/Groovy projects
- Scans BOTH added ("+") and removed ("-") lines (except classes/variables, which track new definitions only)

**Key methodology**:
- **Files**: Detect 3-line anchor (`--- a/`, `+++ b/`, `@@`) to avoid false positives from code lines starting with `--` or `++`
- **Functions**: Scan EVERY line (context/added/removed) + hunk headers (`@@ ... @@ <method>`) to catch methods modified without signature change (e.g., reformatting commits)
- **Classes**: Extract from newly-added lines only
- **Variables**: Extract typed declarations on added lines, supporting Python type hints and Java primitives
- **Imports**: Capture both added and removed imports (a removal still expresses a dependency relationship)
- **Issues**: Only scan the commit-message header region (to avoid false positives in code), identified by `commit ` or `Author:` prefix

In [ ]:
import re
from pathlib import Path

# Real data (after build_diffs.py rebuild): unified-diff "--- a/path"/"+++ b/path"
# pairs followed by "@@" hunk headers, optionally preceded by a synthetic
# "git show" header (commit / Author / Date / message).

_RE_AUTHOR = re.compile(r'^Author:\s+(.+?)\s+<(.+?)>', re.MULTILINE)
_RE_SHOW   = re.compile(r'^(?:commit [0-9a-f]{7,}|Author:\s)', re.MULTILINE)

# Imports — capture BOTH added (+) and removed (-) lines: a commit that
# removes an import still expresses a relationship with that package.
_RE_PY_IMP = re.compile(r'^[+-]\s*(?:from\s+([\w.]+)\s+import|import\s+([\w.]+))', re.MULTILINE)
_RE_JV_IMP = re.compile(r'^[+-]\s*import\s+(?:static\s+)?([\w.]+)\s*;', re.MULTILINE)

_RE_CLASS  = re.compile(r'^\+\s*(?:public\s+|private\s+|protected\s+|abstract\s+|final\s+|static\s+)*(?:class|interface|enum)\s+(\w+)', re.MULTILINE)

# Function signatures — strict modifier requirement keeps false positives
# low even when scanned against EVERY line (added / removed / context).
_RE_PY_FN  = re.compile(r'^\s*(?:async\s+)?def\s+(\w+)\s*\(([^)]*)\)(?:\s*->\s*([^:]+))?')
_RE_JV_FN  = re.compile(r'^\s*(?:(?:public|private|protected|static|final|synchronized|abstract|native|default)\s+)+([\w.<>\[\]]+)\s+(\w+)\s*\(([^)]*)\)')

_RE_HUNK   = re.compile(r'^@@ .*? @@\s*(.*)$', re.MULTILINE)

# Variables — primitives OR any capitalized type (List<T>, Logger, String, …)
_RE_PY_VAR = re.compile(r'^\+\s*(?:self\.)?(\w+)\s*:\s*([\w][\w\[\], .]*?)\s*=', re.MULTILINE)
_RE_JV_VAR = re.compile(
    r'^\+\s*(?:final\s+)?'
    r'(int|long|float|double|boolean|char|byte|short|[A-Z]\w*(?:<[^>]+>)?(?:\[\])?)'
    r'\s+(\w+)\s*[=;]',
    re.MULTILINE)

_RE_ISSUE  = re.compile(r'\b([A-Z][A-Z0-9]+-\d+)\b')


def _strip_prefix(p):
    p = p.strip().strip('"')
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


def parse_files(diff):
    """list of (filepath, 'add'|'remove'|'modify').

    "--- x" / "+++ y" pair followed by an "@@" hunk header. Requiring
    the 3rd-line "@@" avoids matching code lines that happen to start
    with '-- ' / '++ '.
    """
    if not isinstance(diff, str):
        return []
    lines = diff.split("\n")
    out = []
    for i in range(len(lines) - 2):
        l1, l2, l3 = lines[i], lines[i + 1], lines[i + 2]
        if l1.startswith("--- ") and l2.startswith("+++ ") and l3.startswith("@@"):
            a = _strip_prefix(l1[4:])
            b = _strip_prefix(l2[4:])
            if a == "/dev/null":
                out.append((b, "add"))
            elif b == "/dev/null":
                out.append((a, "remove"))
            else:
                out.append((b, "modify"))
    return out


def parse_author(diff):
    """(name, email) or (None, None) — fallback when the row has no author column."""
    if not isinstance(diff, str):
        return None, None
    m = _RE_AUTHOR.search(diff)
    return (m.group(1), m.group(2)) if m else (None, None)


def author_from_row(row, diff=None):
    """Prefer CSV 'author_email' / 'author' / 'author_name' column;
    fall back to parsing the diff's git-show header.
    Returns a string identifier or 'unknown'.
    """
    import pandas as _pd
    if hasattr(row, "get"):
        for col in ("author_email", "author", "author_name"):
            val = row.get(col)
            if val is not None and _pd.notna(val) and str(val).strip():
                return str(val).strip()
    _, email = parse_author(diff)
    return email or "unknown"


def parse_imports(diff):
    """set of top-level package names this diff INTERACTS with (added or removed)."""
    if not isinstance(diff, str):
        return set()
    pkgs = set()
    for m in _RE_PY_IMP.finditer(diff):
        pkg = (m.group(1) or m.group(2) or "").split(".")[0]
        if pkg:
            pkgs.add(pkg)
    for m in _RE_JV_IMP.finditer(diff):
        pkgs.add(m.group(1).split(".")[0])
    return pkgs


def parse_classes(diff):
    if not isinstance(diff, str):
        return []
    return [m.group(1) for m in _RE_CLASS.finditer(diff)]


def _match_fn(text):
    """Return (name, args, ret|None) if `text` holds a Py/Java signature."""
    m = _RE_PY_FN.search(text)
    if m:
        return m.group(1), m.group(2).strip(), (m.group(3) or "").strip() or None
    m = _RE_JV_FN.search(text)
    if m:
        return m.group(2), m.group(3).strip(), m.group(1).strip()
    return None


def parse_functions(diff):
    """list of (name, args, return_type|None) — scans EVERY line of the
    diff (added / removed / context) plus "@@ ... @@ <heading>" section
    headings. Context-line scanning catches methods modified without
    signature change (common in reformatting commits).
    """
    if not isinstance(diff, str):
        return []
    out, seen = [], set()
    for line in diff.splitlines():
        stripped = line[1:] if line and line[0] in "+- " else line
        hit = _match_fn(stripped)
        if hit and hit[0] not in seen:
            seen.add(hit[0]); out.append(hit)
    for m in _RE_HUNK.finditer(diff):
        hit = _match_fn(m.group(1))
        if hit and hit[0] not in seen:
            seen.add(hit[0]); out.append(hit)
    return out


def parse_variables(diff):
    """list of unique (var_name, type_name) for typed declarations ADDED."""
    if not isinstance(diff, str):
        return []
    out, seen = [], set()
    for m in _RE_PY_VAR.finditer(diff):
        v, t = m.group(1), m.group(2).strip()
        if (v, t) not in seen:
            seen.add((v, t)); out.append((v, t))
    for m in _RE_JV_VAR.finditer(diff):
        v, t = m.group(2), m.group(1)
        if (v, t) not in seen:
            seen.add((v, t)); out.append((v, t))
    return out


def parse_issues(diff):
    """set of JIRA-style issue IDs from the commit MESSAGE.

    Only scans the header region of 'git show' output, so code tokens
    are never mistaken for issue keys. Empty for header-less diffs.
    """
    if not isinstance(diff, str) or not _RE_SHOW.search(diff):
        return set()
    header = re.split(r'^(?:diff --git |--- )', diff, maxsplit=1, flags=re.MULTILINE)[0]
    return set(_RE_ISSUE.findall(header))


print("Parser functions ready (context-line method scan, broader var types, +/- imports).")

## Print log of each modification and consumed time

In [ ]:
import time
from collections import Counter

_W = 68   # log column width

def timed_update_kg(G, row, diff, prev_cid=None):
    counts_before = Counter(nx.get_node_attributes(G, 'type').values())
    n_before = G.number_of_nodes()
    e_before = G.number_of_edges()
    t0 = time.perf_counter()
    update_kg(G, row, diff, prev_cid=prev_cid)
    elapsed_ms = (time.perf_counter() - t0) * 1_000
    counts_after = Counter(nx.get_node_attributes(G, 'type').values())
    n_after = G.number_of_nodes()
    e_after = G.number_of_edges()
    all_types = sorted(set(counts_before) | set(counts_after))
    delta = {t: counts_after[t] - counts_before[t]
             for t in all_types if counts_after[t] != counts_before[t]}
    return dict(elapsed_ms=elapsed_ms, nodes_before=n_before, nodes_after=n_after,
                edges_before=e_before, edges_after=e_after,
                delta_types=delta, counts_after=dict(counts_after))


def print_commit_log(row, step, r, total=None):
    lbl = 'BUGGY' if row.buggy else ('FIX  ' if row.fix else 'CLEAN')
    ctr = f'#{step:>4d}' + (f'/{total}' if total else '')
    dn = r['nodes_after'] - r['nodes_before']
    de = r['edges_after'] - r['edges_before']
    print('━' * _W)
    print(f'  {ctr}  {row.commit_id[:11]}…  {lbl}  {int(row.year)}'
          f'   {r["elapsed_ms"]:7.2f} ms')
    print('─' * _W)
    print(f'  Nodes : {r["nodes_before"]:>6,} → {r["nodes_after"]:>6,}  ({dn:+d})'
          f'    Edges : {r["edges_before"]:>7,} → {r["edges_after"]:>7,}  ({de:+d})')
    if r['delta_types']:
        print(f'  {"─" * 54}')
        print(f'  {"Node type":<26}  {"Added":>6}  {"Total":>8}')
        print(f'  {"─" * 54}')
        for t in sorted(r['delta_types'], key=lambda x: -r['delta_types'][x]):
            print(f'  {t:<26}  {r["delta_types"][t]:>+6d}  {r["counts_after"].get(t,0):>8,}')
    print()


def print_batch_summary(label, results_list, df_slice):
    if not results_list:
        return
    total_ms = sum(r['elapsed_ms'] for r in results_list)
    avg_ms = total_ms / len(results_list)
    worst = max(results_list, key=lambda r: r['elapsed_ms'])
    dn_total = sum(r['nodes_after'] - r['nodes_before'] for r in results_list)
    de_total = sum(r['edges_after'] - r['edges_before'] for r in results_list)
    n_final = results_list[-1]['nodes_after']
    e_final = results_list[-1]['edges_after']
    n_buggy = int(df_slice.buggy.sum())
    agg = Counter()
    for r in results_list:
        agg.update(r['delta_types'])
    print('=' * _W)
    print(f'  BATCH SUMMARY — {label}')
    print('=' * _W)
    print(f'  Commits processed : {len(results_list):,}'
          f'   ({n_buggy} buggy / {len(results_list) - n_buggy} clean)')
    print(f'  Total time        : {total_ms/1_000:,.3f} s'
          f'   (avg {avg_ms:.1f} ms / commit,'
          f'   slowest {worst["elapsed_ms"]:.1f} ms)')
    print(f'  Nodes added       : {dn_total:+,}   → graph total: {n_final:,}')
    print(f'  Edges added       : {de_total:+,}   → graph total: {e_final:,}')
    if agg:
        final_counts = results_list[-1]['counts_after']
        print(f'  {"─" * 54}')
        print(f'  {"Node type":<26}  {"Added":>7}  {"Total now":>10}')
        print(f'  {"─" * 54}')
        for t in sorted(agg, key=lambda x: -agg[x]):
            print(f'  {t:<26}  {agg[t]:>+7,}  {final_counts.get(t, 0):>10,}')
    print()

print('timed_update_kg / print_commit_log / print_batch_summary  ready.')

## 3 — Graph Update Function

The core of the online simulation: `update_kg(G, row, diff, prev_cid)` adds **one commit's knowledge** to the graph.
Called **after** prediction (step 4 of the online loop), it implements the **5-tier KG model**:

| Tier | Entities | Source |
|------|----------|--------|
| **1 — Core** | COMMIT, TIME, INTERVAL, LABEL | CSV (timestamps, labels) |
| **2 — Author** | AUTHOR, AUTHOR_INTERVAL | diff_text header (Author: email) or CSV author column |
| **2 — File** | FILE, DIR, FILE_TYPE, EXTERNAL_PACKAGE | parse_files, parse_imports |
| **4 — Within-file** | CLASS, FUNCTION, FUNCTION_SIGNATURE, VARIABLE, DATA_TYPE | parse_classes, parse_functions, parse_variables |
| **5 — Semantic** | ISSUE, BRANCH | commit_issue_map + issue_texts CSV, or git repo queries |

**Per-entity counters** (attached as node attributes):
- **AUTHOR**: commit_count, bug_count (cumulative)
- **FILE**: change_count, bug_count (cumulative), authors (set of email addresses)

**Intervals** track temporal spans: COMMIT→INTERVAL→TIME relations form a timeline of activity per author and per file.

In [ ]:
# ── BRANCH helper ─────────────────────────────────────────────────────
try:
    from git import Repo as _Repo
    _GIT_OK = True
except ModuleNotFoundError:
    _Repo = None
    _GIT_OK = False
    print("[branch] gitpython not installed — BRANCH tier will be empty.")

_branch_cache = {}

def _build_branch_map(repo):
    """
    Build {commit_sha: set(branch_names)} using first-parent traversal.

    Walks main/master first; for every other branch stops when it hits a
    commit already in the main lineage — prevents the root commit from
    appearing in all branches.  Indexed at full + 12-char SHA.
    """
    sha_map = defaultdict(set)
    _main_names = {"master", "main", "trunk", "develop"}

    all_refs = list(repo.heads)
    for remote in repo.remotes:
        for ref in remote.refs:
            if not ref.name.endswith("/HEAD"):
                all_refs.append(ref)

    main_ref = next((r for r in all_refs if r.name.split("/")[-1] in _main_names), None)
    if main_ref is None and all_refs:
        main_ref = all_refs[0]

    main_shas = set()
    if main_ref:
        for c in repo.iter_commits(main_ref, first_parent=True):
            main_shas.add(c.hexsha)
            sha_map[c.hexsha].add(main_ref.name)
            sha_map[c.hexsha[:12]].add(main_ref.name)

    for br in all_refs:
        if main_ref and br.name == main_ref.name:
            continue
        try:
            for c in repo.iter_commits(br, first_parent=True):
                if c.hexsha in main_shas:
                    break
                sha_map[c.hexsha].add(br.name)
                sha_map[c.hexsha[:12]].add(br.name)
        except Exception:
            continue

    return sha_map


def commit_branches(project, commit_sha):
    """Branches whose first-parent lineage directly contains commit_sha."""
    if not _GIT_OK:
        return []
    if project not in _branch_cache:
        slug = str(project).split("/")[-1]
        repo_path = REPOS_DIR / slug
        if not repo_path.exists():
            _branch_cache[project] = None
            print(f"  [branch] repo missing: {repo_path}")
        else:
            try:
                _branch_cache[project] = _build_branch_map(_Repo(repo_path))
                print(f"  [branch] {project}: indexed {len(_branch_cache[project]):,} commits")
            except Exception as e:
                print(f"  [branch] {project}: failed — {e}")
                _branch_cache[project] = None
    bm = _branch_cache.get(project)
    if not bm:
        return []
    for key in (commit_sha, commit_sha[:12], commit_sha[:7]):
        result = bm.get(key)
        if result:
            return sorted(result)
    return []


# ── Interval helpers ───────────────────────────────────────────────────
#
# Interval model
# ──────────────
# Exactly 6 node types have an interval node:
#   COMMIT, FILE, CLASS, FUNCTION, FUNCTION_SIGNATURE, BRANCH
#
# Each gets ONE interval node linked via -[has_interval]-> INTERVAL.
# The interval node has two attributes:
#   begin : float  — Unix timestamp of the entity first appearance (never changes)
#   end   : float  — Unix timestamp of the most recent modification; float("inf")
#                    until the entity is touched again for the first time
#
# When the entity is created:       _init_interval(G, eid, ts)
# When the entity is touched again: _update_interval_end(G, eid, ts)
# ──────────────────────────────────────────────────────────────────────
_itvl_seq = [0]


def _init_interval(G, entity_id, ts):
    """Create the single interval node for entity_id (begin=ts, end=inf)."""
    _itvl_seq[0] += 1
    iid = f"itvl:{_itvl_seq[0]}"
    G.add_node(iid, type="INTERVAL", begin=float(ts), end=float("inf"))
    G.add_edge(entity_id, iid, rel="has_interval")
    G.nodes[entity_id]["_cur_itvl"] = iid
    return iid


def _update_interval_end(G, entity_id, ts):
    """Update end of the entity single interval to ts (most recent modification)."""
    cur = G.nodes[entity_id].get("_cur_itvl")
    if cur and G.has_node(cur):
        G.nodes[cur]["end"] = float(ts)


# ── update_kg ──────────────────────────────────────────────────────────
def update_kg(G, row, diff, prev_cid=None):
    """
    Add one commit knowledge to G.  Call AFTER prediction (step 4 of loop).

    Interval nodes (one per entity, begin fixed, end updated on each touch):
      COMMIT            — begin = commit timestamp, end stays inf (unique events)
      FILE              — begin = first seen, end = latest commit that touches it
      CLASS             — begin = first seen, end = latest commit that touches it
      FUNCTION          — begin = first seen, end = latest commit that touches it
      FUNCTION_SIGNATURE — begin = first seen, end = latest commit that touches it
      BRANCH            — begin = first commit on branch, end = latest commit on it

    All other node types (AUTHOR, DIR, LABEL, ISSUE, …) have no interval node.
    """
    cid = f"commit:{row.commit_id}"
    ts  = float(row.author_date)

    # ── TIER 1: Core ──────────────────────────────────────────────────

    G.add_node(cid, type="COMMIT", commit_id=row.commit_id,
               project=row.project, year=int(row.year), author_date=ts)
    _init_interval(G, cid, ts)   # commits are unique — end stays inf

    tid = f"time:{int(ts)}"
    if not G.has_node(tid):
        G.add_node(tid, type="TIME", datetime=ts)
    G.add_edge(cid, tid, rel="at_time")

    if prev_cid is not None:
        G.add_edge(prev_cid, cid, rel="precedes")

    status = -1 if row.buggy else (1 if row.fix else 0)
    lid = f"label:{status}"
    if not G.has_node(lid):
        G.add_node(lid, type="LABEL", status=status)
    G.add_edge(cid, lid, rel="is")

    # ── TIER 2: Author ────────────────────────────────────────────────

    email = author_from_row(row, diff)
    aid   = f"author:{email}"
    if not G.has_node(aid):
        G.add_node(aid, type="AUTHOR", email=email, commit_count=0, bug_count=0)
    G.add_edge(cid, aid, rel="by")
    G.nodes[aid]["commit_count"] += 1
    if row.buggy:
        G.nodes[aid]["bug_count"] += 1

    # ── TIER 2: Files, dirs, file types ──────────────────────────────

    files = parse_files(diff)
    for filepath, change_type in files:
        p    = Path(filepath)
        fid  = f"file:{filepath}"
        ftid = f"filetype:{p.suffix or 'none'}"

        if not G.has_node(fid):
            G.add_node(fid, type="FILE", name=p.name, path=filepath,
                       change_count=0, bug_count=0, authors=set())
            _init_interval(G, fid, ts)
        else:
            _update_interval_end(G, fid, ts)
        G.add_edge(cid, fid, rel=change_type)
        G.nodes[fid]["change_count"] += 1
        if row.buggy:
            G.nodes[fid]["bug_count"] += 1
        G.nodes[fid]["authors"].add(email)

        if not G.has_node(ftid):
            G.add_node(ftid, type="FILE_TYPE", format=p.suffix)
        G.add_edge(fid, ftid, rel="type")

        parents = list(reversed(list(p.parents)))
        for i, part in enumerate(parents):
            if str(part) in (".", ""):
                continue
            did = f"dir:{part}"
            if not G.has_node(did):
                G.add_node(did, type="DIR", name=part.name, path=str(part))
            if i == len(parents) - 1:
                G.add_edge(fid, did, rel="parent")
            if i > 0:
                pdid = f"dir:{parents[i - 1]}"
                if G.has_node(pdid):
                    G.add_edge(did, pdid, rel="parent")
                    G.add_edge(pdid, did, rel="child")

    # ── TIER 3: External packages ─────────────────────────────────────

    for pkg in parse_imports(diff):
        pid = f"extpkg:{pkg}"
        if not G.has_node(pid):
            G.add_node(pid, type="EXTERNAL_PACKAGE", name=pkg)
        for fp, _ in files:
            G.add_edge(f"file:{fp}", pid, rel="imports")

    # ── TIER 4: Within-file entities ──────────────────────────────────

    fids = [f"file:{fp}" for fp, _ in files]

    for cls in parse_classes(diff):
        clid = f"class:{cls}"
        if not G.has_node(clid):
            G.add_node(clid, type="CLASS", name=cls)
            _init_interval(G, clid, ts)
        else:
            _update_interval_end(G, clid, ts)
        for fid in fids:
            G.add_edge(fid, clid, rel="contains")

    for fn, args, ret in parse_functions(diff):
        fnid   = f"func:{fn}"
        sig_id = f"sig:{fn}({args})->{ret}"
        if not G.has_node(fnid):
            G.add_node(fnid, type="FUNCTION", name=fn)
            _init_interval(G, fnid, ts)
        else:
            _update_interval_end(G, fnid, ts)
        if not G.has_node(sig_id):
            G.add_node(sig_id, type="FUNCTION_SIGNATURE", args=args, returns=ret)
            _init_interval(G, sig_id, ts)
        else:
            _update_interval_end(G, sig_id, ts)
        G.add_edge(fnid, sig_id, rel="has_signature")
        for fid in fids:
            G.add_edge(fid, fnid, rel="contains")

    for var, dtype in parse_variables(diff):
        vid  = f"var:{var}"
        dtid = f"dtype:{dtype}"
        if not G.has_node(vid):
            G.add_node(vid, type="VARIABLE", name=var)
        if not G.has_node(dtid):
            G.add_node(dtid, type="DATA_TYPE", id=dtype)
        G.add_edge(vid, dtid, rel="has_type")
        for fid in fids:
            G.add_edge(fid, vid, rel="contains")

    # ── TIER 5: Issues ────────────────────────────────────────────────

    issue_ids = list(commit_to_issues.get(row.commit_id, []))
    if not issue_ids:
        issue_ids = sorted(parse_issues(diff))
    for issue_id in issue_ids:
        inode = f"issue:{issue_id}"
        if not G.has_node(inode):
            meta = issue_meta.get(issue_id, {})
            G.add_node(inode, type="ISSUE", id=issue_id,
                       title       = meta.get("title", ""),
                       description = meta.get("description", ""),
                       status      = meta.get("status", ""),
                       created     = meta.get("created", ""),
                       updated     = meta.get("updated", ""))
        G.add_edge(cid, inode, rel="for")

    # ── TIER 5: Branch ────────────────────────────────────────────────
    # BRANCH node created on first commit arrival; end updated on each new commit.

    for br in commit_branches(row.project, row.commit_id):
        bid = f"branch:{br}"
        if not G.has_node(bid):
            G.add_node(bid, type="BRANCH", id=br)
            _init_interval(G, bid, ts)
        else:
            _update_interval_end(G, bid, ts)
        G.add_edge(cid, bid, rel="in")


print("update_kg() ready.")
print("  Interval nodes : COMMIT · FILE · CLASS · FUNCTION · FUNCTION_SIGNATURE · BRANCH")
print("  Each entity    : one interval, begin=first_seen (fixed), end=last_touch (updated)")
print("  Branch model   : first-parent traversal, stops at main lineage, 3-length SHA fallback.")


## 3.1 — Parser Sanity Test

Applies every parser to a **real** `apache_groovy` diff sample so you can inspect exactly what each one extracts.
This validates parser behavior **before** the full pipeline runs, serving as both documentation and early-warning
for regex failures on real data. The cell also demonstrates how `update_kg()` translates parsed data into KG nodes and edges.

In [ ]:
# Real sample from the apache_groovy diff_text column (GitPython create_patch).
SAMPLE_DIFF = r"""--- a/src/main/org/codehaus/groovy/runtime/DefaultGroovyMethods.java
+++ b/src/main/org/codehaus/groovy/runtime/DefaultGroovyMethods.java
@@ -65,7 +65,6 @@ STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE)
 import java.sql.ResultSet;
 import java.sql.SQLException;
 import java.util.ArrayList;
-import java.util.Arrays;
 import java.util.Collection;
@@ -655,23 +654,23 @@ public static List reverse(List self) {
     public static List plus(List left, Collection right) {
-            List answer = new ArrayList(left.size()+right.size());
+        List answer = new ArrayList(left.size() + right.size());
@@ -679,14 +678,15 @@ public static List multiply(List self, Number factor) {
     public static List intersect(List left, Collection right) {
-        if (left.size()==0)
+        if (left.size() == 0)
@@ -698,7 +698,7 @@ public static List intersect(List left, Collection right) {
     public static List minus(List self, Collection removeMe) {
-        if (self.size() ==0 )
+        if (self.size() == 0 )
@@ -728,13 +729,15 @@ public static List minus(List self, Collection removeMe) {
-           List answer=new LinkedList();
+           List answer = new LinkedList();
@@ -751,7 +754,7 @@ public static List flatten(List self) {
-            Object element=iter.next();
+            Object element = iter.next();
@@ -767,23 +770,23 @@ else if (element instanceof Map) {
     private static boolean sameType(Collection[] cols)
@@ -867,11 +870,14 @@ else if (isLong(left) || isLong(right)) {
     public static Number power(Number self, Number exponent) {
-        double answer=Math.pow(self.doubleValue(), exponent.doubleValue());
+        double answer = Math.pow(self.doubleValue(),
+                exponent.doubleValue());
-        int size=factor.intValue();
+        int size = factor.intValue();
"""

print("=" * 64)
print("PARSER SANITY TEST  (on a real apache_groovy diff sample)")
print("=" * 64)

print("\nparse_files:")
for fp, ch in parse_files(SAMPLE_DIFF):
    print(f"   [{ch:6}] {fp}")

print("\nparse_author:")
print("  ", parse_author(SAMPLE_DIFF), " <- (None, None) expected: raw diff has no header")

print("\nparse_imports (added '+import' only):")
print("  ", parse_imports(SAMPLE_DIFF) or "set()  <- sample only removes an import")

print("\nparse_classes:")
print("  ", parse_classes(SAMPLE_DIFF) or "[]  <- no class added in sample")

print("\nparse_functions (added lines + '@@ ...@@ heading' enclosing methods):")
for nm, args, ret in parse_functions(SAMPLE_DIFF):
    print(f"   {nm:12} args=({args})  ret={ret}")

print("\nparse_variables (typed declarations on added lines):")
for v, t in parse_variables(SAMPLE_DIFF):
    print(f"   {t:8} {v}")

print("\nparse_issues:")
print("  ", parse_issues(SAMPLE_DIFF) or "set()  <- no commit message in raw diff")

# ── What update_kg would add for this single commit ───────────────────
print("\n" + "=" * 64)
print("update_kg() DRY RUN on the sample")
print("=" * 64)
_g = nx.MultiDiGraph()
_row = pd.Series({
    "commit_id": "deadbeefcafe", "project": "apache/groovy",
    "buggy": False, "fix": True, "year": 2005, "author_date": 1111111111,
})
update_kg(_g, _row, SAMPLE_DIFF, prev_cid=None)
print(pd.Series(nx.get_node_attributes(_g, "type")).value_counts().to_string())
print(f"\n  total: {_g.number_of_nodes()} nodes, {_g.number_of_edges()} edges")

In [ ]:
def extract_kg_features(G, row, diff):
    """
    Query G for features about this commit BEFORE it is added.
    All values default to 0.0 if the relevant history does not exist yet.
    """
    feats = {}

    # ── Project-level ─────────────────────────────────────────────────
    commit_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "COMMIT"]
    n = len(commit_nodes)
    feats["kg_project_commit_count"] = float(n)
    if n > 0:
        n_bug = sum(
            1 for cn in commit_nodes
            if any(d.get("rel") == "is" and v == "label:-1"
                   for _, v, d in G.out_edges(cn, data=True))
        )
        feats["kg_project_bug_rate"] = n_bug / n
    else:
        feats["kg_project_bug_rate"] = 0.0

    # ── Author-level ──────────────────────────────────────────────────
    # Same identifier source as update_kg → consistent AUTHOR-node lookup.
    email = author_from_row(row, diff)
    aid = f"author:{email}"
    ad  = G.nodes[aid] if G.has_node(aid) else {}
    ac  = ad.get("commit_count", 0)
    bc  = ad.get("bug_count",    0)
    feats["kg_author_commit_count"] = float(ac)
    feats["kg_author_bug_rate"]     = bc / ac if ac > 0 else 0.0

    # ── File-level ────────────────────────────────────────────────────
    change_counts, bug_rates, uniq_authors = [], [], []
    for fp, _ in parse_files(diff):
        fid = f"file:{fp}"
        if not G.has_node(fid):
            continue
        fd  = G.nodes[fid]
        cc  = fd.get("change_count", 0)
        bcc = fd.get("bug_count",    0)
        change_counts.append(cc)
        bug_rates.append(bcc / cc if cc > 0 else 0.0)
        uniq_authors.append(len(fd.get("authors", set())))

    feats["kg_file_change_count"]   = float(sum(change_counts))
    feats["kg_file_bug_rate"]       = sum(bug_rates)   / len(bug_rates)   if bug_rates   else 0.0
    feats["kg_file_unique_authors"] = sum(uniq_authors) / len(uniq_authors) if uniq_authors else 0.0

    return feats


print("extract_kg_features() ready.")

## 5 — Online Simulation Loop

This is the **heart** of the online learning pipeline. Commits arrive in chronological order (timestamp-sorted).
For **each commit**, the 4-step cycle executes:

1. **Extract KG features** — query G, which contains only commits **strictly before** this one
2. **Predict** — use SGDClassifier to predict bug/clean label (after WARMUP=50 commits)
3. **Record result** — store true_label and prediction for later evaluation
4. **Update KG** — add this commit's entities and edges to G
5. **Online model update** — call model.partial_fit and scaler.partial_fit with this commit's true label

This ensures **zero label leakage**: predictions are made using only historical KG state.
After WARMUP, the model trains continuously as new commits arrive, with features dynamically enriched by the growing KG.

In [ ]:
# Handcrafted features available from the CSV
HC_COLS = ["la", "ld", "nf", "nd", "ns", "ent", "ndev", "age", "nuc", "aexp", "arexp", "asexp"]

# KG features in fixed order
KG_COLS = [
    "kg_project_commit_count",
    "kg_project_bug_rate",
    "kg_author_commit_count",
    "kg_author_bug_rate",
    "kg_file_change_count",
    "kg_file_bug_rate",
    "kg_file_unique_authors",
]

# ── Initialise empty graph and model ─────────────────────────────────
G            = nx.MultiDiGraph()
model        = SGDClassifier(loss="log_loss", max_iter=1, warm_start=True, random_state=42)
scaler       = StandardScaler()
results      = []          # accumulates {commit_id, true_label, predicted}
prev_cid_map = {}          # project → node-id of last commit seen
WARMUP       = 50          # commits before we start predicting

print(f"Starting online simulation over {len(df):,} commits …")

## 4 — Base KG: Pre-Dataset Project History

Before the online loop starts, we build an initial KG from all git commits
that **predate the dataset window** (`df.author_date.min()`).  This gives the
model prior knowledge about files, functions, authors, and branches that
already existed before the defect-tracking period began — the "commit 0" state.

**How it works**:
1. Walk the local git clone's `first_parent` history from `HEAD` backwards
2. Collect every commit whose `committed_date` is earlier than the dataset start
3. Feed each through `update_kg()` in chronological order (oldest first)
4. Store the last commit ID so the main loop chains from it correctly

`MAX_PRE_COMMITS` limits how many recent pre-dataset commits are processed
(set to `None` to process the full project history — may take several minutes).

In [ ]:
def build_base_kg(G, project, repo_path, until_ts, max_commits=2000):
    """
    Populate G with git commits that predate until_ts (Unix seconds).

    Parameters
    ----------
    G           : nx.MultiDiGraph — graph to populate (modified in place)
    project     : str  — project slug matching row.project in the CSV (e.g. "apache/groovy")
    repo_path   : Path — path to the local git clone
    until_ts    : float — earliest timestamp in the dataset; only earlier commits are used
    max_commits : int | None — cap on commits processed (None = all pre-dataset history)

    Returns
    -------
    last_cid : str | None — node-id of the final pre-dataset commit (for chaining)
    n        : int        — number of commits processed
    """
    if not _GIT_OK:
        print("[base KG] skipped — GitPython not installed.")
        return None, 0
    if not repo_path.exists():
        print(f"[base KG] skipped — repo not found at {repo_path}")
        return None, 0

    try:
        repo = _Repo(repo_path)
    except Exception as e:
        print(f"[base KG] cannot open repo: {e}")
        return None, 0

    # Collect commits before until_ts via first-parent walk (chronologically stable)
    pre_commits = []
    for c in repo.iter_commits('HEAD', first_parent=True):
        if c.committed_date < until_ts:
            pre_commits.append(c)
    pre_commits.reverse()   # oldest first

    cap_label = ""
    if max_commits is not None and len(pre_commits) > max_commits:
        pre_commits = pre_commits[-max_commits:]   # keep most-recent N pre-dataset commits
        cap_label = f"  (capped at most-recent {max_commits:,})"

    total = len(pre_commits)
    if total == 0:
        print("[base KG] no pre-dataset commits found — G starts empty.")
        return None, 0

    first_dt = pre_commits[0].committed_datetime.strftime('%Y-%m-%d')
    last_dt  = pre_commits[-1].committed_datetime.strftime('%Y-%m-%d')
    print(f"[base KG] {total:,} pre-dataset commits{cap_label}")
    print(f"          {first_dt}  →  {last_dt}  (before dataset window)")
    print(f"[base KG] building …")

    prev_cid = None
    last_cid = None
    for i, c in enumerate(pre_commits):
        row = pd.Series({
            'commit_id':   c.hexsha,
            'project':     project,
            'buggy':       False,
            'fix':         False,
            'year':        int(c.committed_datetime.year),
            'author_date': float(c.committed_date),
        })
        try:
            diff_text = repo.git.show(c.hexsha, '--format=', '-p')
        except Exception:
            diff_text = None

        update_kg(G, row, diff_text, prev_cid=prev_cid)
        prev_cid = f"commit:{c.hexsha}"
        last_cid = prev_cid

        if (i + 1) % 500 == 0:
            print(f"  {i+1:,} / {total:,}  —  {G.number_of_nodes():,} nodes, "
                  f"{G.number_of_edges():,} edges")

    print(f"[base KG] done.  {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    return last_cid, total


# ── Run ────────────────────────────────────────────────────────────────
MAX_PRE_COMMITS = 2000   # set None to process the entire pre-dataset history

_base_last_cid, _n_base = build_base_kg(
    G,
    project     = f"apache/{REPO_SLUG}",
    repo_path   = REPO_PATH,
    until_ts    = float(df.author_date.min()),
    max_commits = MAX_PRE_COMMITS,
)

# Pre-seed prev_cid_map so the main loop chains directly from the base KG
if _base_last_cid:
    prev_cid_map[f"apache/{REPO_SLUG}"] = _base_last_cid

print(f"\nBase KG  : {_n_base:,} pre-dataset commits processed")
print(f"Graph    : {G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges")
print(f"Dataset  : {len(df):,} commits will now replay on top of this base state")

In [ ]:
# ── Base KG Visualization ──────────────────────────────────────────────
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict

_NC = {
    'COMMIT':             '#4C72B0',
    'TIME':               '#55A868',
    'INTERVAL':           '#FF8C00',
    'LABEL':              '#C44E52',
    'AUTHOR':             '#8172B2',
    'FILE':               '#937860',
    'FILE_TYPE':          '#DA8BC3',
    'DIR':                '#8C8C8C',
    'EXTERNAL_PACKAGE':   '#CCB974',
    'CLASS':              '#64B5CD',
    'FUNCTION':           '#1F77B4',
    'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE':           '#98DF8A',
    'DATA_TYPE':          '#FF9896',
    'ISSUE':              '#FFBB78',
    'BRANCH':             '#17BECF',
}

def _grp(G):
    g = defaultdict(list)
    for n, d in G.nodes(data=True):
        g[d.get('type', 'UNKNOWN')].append(n)
    return g

def _sh(nid, mx=18):
    s = nid.split(':', 1)[-1]
    return s[:mx] + '...' if len(s) > mx else s

# ── Bar charts: node types & edge relations ────────────────────────────
node_types = Counter(d.get('type', '?') for _, d in G.nodes(data=True))
edge_rels  = Counter(d.get('rel',  '?') for _, _, d in G.edges(data=True))

nt_lab, nt_cnt = zip(*sorted(node_types.items(), key=lambda x: -x[1]))
er_lab, er_cnt = zip(*sorted(edge_rels.items(),  key=lambda x: -x[1]))

go.Figure(
    go.Bar(x=nt_cnt, y=nt_lab, orientation='h',
           marker_color=[_NC.get(t, '#DDD') for t in nt_lab],
           text=nt_cnt, textposition='outside'),
    layout=go.Layout(
        title=f'Base KG — Node types  ({G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges)',
        xaxis_title='Count', height=380,
        margin=dict(l=160, r=60, t=45, b=40),
        yaxis=dict(autorange='reversed'),
    )
).show()

go.Figure(
    go.Bar(x=er_cnt, y=er_lab, orientation='h',
           marker_color='#5B9BD5',
           text=er_cnt, textposition='outside'),
    layout=go.Layout(
        title='Base KG — Edge relation types',
        xaxis_title='Count', height=350,
        margin=dict(l=160, r=60, t=45, b=40),
        yaxis=dict(autorange='reversed'),
    )
).show()

# ── Network: sample if large ───────────────────────────────────────────
_MAX_VIZ = 300
if G.number_of_nodes() <= _MAX_VIZ:
    G_viz = G
    _vtitle = 'Base KG'
else:
    seeds = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)[:40]
    vnodes = set(seeds)
    for s in seeds:
        for nb in list(G.successors(s)) + list(G.predecessors(s)):
            vnodes.add(nb)
            if len(vnodes) >= _MAX_VIZ:
                break
        if len(vnodes) >= _MAX_VIZ:
            break
    G_viz = G.subgraph(list(vnodes)[:_MAX_VIZ]).copy()
    _vtitle = f'Base KG  (sampled {G_viz.number_of_nodes()} of {G.number_of_nodes():,} nodes)'

_n = G_viz.number_of_nodes()
_pos = nx.spring_layout(G_viz, dim=3, seed=42, k=2.0/(_n**0.5), iterations=60)

ex, ey, ez = [], [], []
for u, v in G_viz.edges():
    x0,y0,z0 = _pos[u]; x1,y1,z1 = _pos[v]
    ex += [x0,x1,None]; ey += [y0,y1,None]; ez += [z0,z1,None]

traces = [go.Scatter3d(x=ex, y=ey, z=ez, mode='lines',
                       line=dict(color='#AAA', width=1),
                       hoverinfo='none', showlegend=False)]

sz = max(3, 10 - _n // 20)
for t, ns in sorted(_grp(G_viz).items()):
    traces.append(go.Scatter3d(
        x=[_pos[nd][0] for nd in ns],
        y=[_pos[nd][1] for nd in ns],
        z=[_pos[nd][2] for nd in ns],
        mode='markers',
        marker=dict(size=sz, color=_NC.get(t, '#DDD'),
                    opacity=0.88, line=dict(width=0.5, color='#333')),
        text=[_sh(nd, 30) for nd in ns],
        hoverinfo='text+name',
        name=f'{t} ({len(ns)})',
    ))

_ax = dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False)
go.Figure(
    data=traces,
    layout=go.Layout(
        title=f'{_vtitle}  |  nodes={_n}  edges={G_viz.number_of_edges()}',
        scene=dict(xaxis=_ax, yaxis=_ax, zaxis=_ax),
        margin=dict(l=0, r=0, b=0, t=45), height=550,
        legend=dict(font=dict(size=9), itemsizing='constant'),
    )
).show()


In [ ]:
# ── Base KG — Full visualization (all nodes, cluster layout) ──────────
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict

_NC = {
    'COMMIT':             '#4C72B0',
    'TIME':               '#55A868',
    'INTERVAL':           '#FF8C00',
    'LABEL':              '#C44E52',
    'AUTHOR':             '#8172B2',
    'FILE':               '#937860',
    'FILE_TYPE':          '#DA8BC3',
    'DIR':                '#8C8C8C',
    'EXTERNAL_PACKAGE':   '#CCB974',
    'CLASS':              '#64B5CD',
    'FUNCTION':           '#1F77B4',
    'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE':           '#98DF8A',
    'DATA_TYPE':          '#FF9896',
    'ISSUE':              '#FFBB78',
    'BRANCH':             '#17BECF',
}

def _sh(nid, mx=22):
    s = nid.split(':', 1)[-1]
    return s[:mx] + '…' if len(s) > mx else s

# Group nodes by type
_grp = defaultdict(list)
for nd, d in G.nodes(data=True):
    _grp[d.get('type', 'UNKNOWN')].append(nd)

types = sorted(_grp.keys())
K = len(types)

# Place each type cluster center evenly on a sphere (golden spiral)
rng = np.random.default_rng(42)
centers = {}
golden = np.pi * (3 - np.sqrt(5))
for i, t in enumerate(types):
    y  = 1 - (i / max(K - 1, 1)) * 2
    r  = np.sqrt(max(1 - y*y, 0))
    th = golden * i
    centers[t] = np.array([r * np.cos(th), r * np.sin(th), y]) * 5.0

# Assign each node a position: cluster center + small random offset
node_pos = {}
for t, nodes in _grp.items():
    n   = len(nodes)
    # Random points in a sphere of radius proportional to sqrt(n)
    rad = max(0.4, np.sqrt(n) * 0.06)
    phi   = rng.uniform(0, 2*np.pi, n)
    costh = rng.uniform(-1, 1, n)
    sinth = np.sqrt(1 - costh**2)
    r     = rad * rng.uniform(0, 1, n) ** (1/3)
    offsets = np.column_stack([r*sinth*np.cos(phi),
                               r*sinth*np.sin(phi),
                               r*costh])
    for nd, off in zip(nodes, offsets):
        node_pos[nd] = centers[t] + off

# Build traces — one per node type
traces = []
for t in types:
    nodes = _grp[t]
    xs = [node_pos[nd][0] for nd in nodes]
    ys = [node_pos[nd][1] for nd in nodes]
    zs = [node_pos[nd][2] for nd in nodes]
    sz = max(2, min(6, 120 // max(len(nodes), 1) + 2))
    traces.append(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='markers',
        marker=dict(size=sz, color=_NC.get(t, '#DDD'),
                    opacity=0.80, line=dict(width=0.3, color='#333')),
        text=[_sh(nd, 30) for nd in nodes],
        hoverinfo='text+name',
        name=f'{t} ({len(nodes):,})',
    ))

_ax = dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False)
go.Figure(
    data=traces,
    layout=go.Layout(
        title=(f'Base KG — all nodes  '
               f'({G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges)  '
               f'[cluster layout by node type]'),
        scene=dict(xaxis=_ax, yaxis=_ax, zaxis=_ax),
        margin=dict(l=0, r=0, b=0, t=50), height=620,
        legend=dict(font=dict(size=9), itemsizing='constant'),
    )
).show()


In [ ]:
# ── Base KG — Full visualization, temporal layout (all nodes + edges) ─
# No graph algorithm. COMMIT nodes placed by author_date along x-axis.
# All other nodes placed by mean timestamp of their commit neighbours.
# O(n) computation — works for any graph size.
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict

_NC = {
    'COMMIT':             '#4C72B0',
    'TIME':               '#55A868',
    'INTERVAL':           '#FF8C00',
    'LABEL':              '#C44E52',
    'AUTHOR':             '#8172B2',
    'FILE':               '#937860',
    'FILE_TYPE':          '#DA8BC3',
    'DIR':                '#8C8C8C',
    'EXTERNAL_PACKAGE':   '#CCB974',
    'CLASS':              '#64B5CD',
    'FUNCTION':           '#1F77B4',
    'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE':           '#98DF8A',
    'DATA_TYPE':          '#FF9896',
    'ISSUE':              '#FFBB78',
    'BRANCH':             '#17BECF',
}

_TYPE_Y = {
    'COMMIT':             0.0,
    'BRANCH':             0.8,
    'AUTHOR':             1.6,
    'FILE':               2.4,
    'DIR':                3.0,
    'FILE_TYPE':          3.4,
    'CLASS':              4.2,
    'FUNCTION':           5.0,
    'FUNCTION_SIGNATURE': 5.8,
    'VARIABLE':           6.4,
    'DATA_TYPE':          6.8,
    'EXTERNAL_PACKAGE':   7.4,
    'ISSUE':              8.0,
    'INTERVAL':           8.8,
    'LABEL':              9.4,
    'TIME':               9.8,
}

def _sh(nid, mx=25):
    s = nid.split(':', 1)[-1]
    return s[:mx] + '…' if len(s) > mx else s

rng = np.random.default_rng(42)

# Step 1: assign timestamp to every node
commit_ts = {
    nd: d.get('author_date', d.get('begin', None))
    for nd, d in G.nodes(data=True)
    if d.get('type') in ('COMMIT', 'INTERVAL')
}
node_ts = dict(commit_ts)
for nd in G.nodes():
    if nd in node_ts:
        continue
    neighbours = list(G.successors(nd)) + list(G.predecessors(nd))
    ts_vals = [node_ts[nb] for nb in neighbours if nb in node_ts]
    node_ts[nd] = float(np.mean(ts_vals)) if ts_vals else 0.0

ts_vals_all = [v for v in node_ts.values() if v and v > 0]
ts_min, ts_max = min(ts_vals_all), max(ts_vals_all)
ts_range = max(ts_max - ts_min, 1)

def _tx(nd):
    return (node_ts.get(nd, ts_min) - ts_min) / ts_range

# Step 2: assign final (x, y, z) to every node
_grp = defaultdict(list)
for nd, d in G.nodes(data=True):
    _grp[d.get('type', 'UNKNOWN')].append(nd)

node_xyz = {}
for t, nodes in _grp.items():
    base_y = _TYPE_Y.get(t, 10.0)
    jy = rng.uniform(-0.25, 0.25, len(nodes))
    jz = rng.uniform(-0.3,  0.3,  len(nodes))
    for nd, dy, dz in zip(nodes, jy, jz):
        node_xyz[nd] = (_tx(nd), base_y + dy, float(dz))

# Step 3: edge trace (all edges as lines with None separators)
ex, ey, ez = [], [], []
for u, v in G.edges():
    if u in node_xyz and v in node_xyz:
        x0,y0,z0 = node_xyz[u]
        x1,y1,z1 = node_xyz[v]
        ex += [x0, x1, None]
        ey += [y0, y1, None]
        ez += [z0, z1, None]

traces = [go.Scatter3d(
    x=ex, y=ey, z=ez,
    mode='lines',
    line=dict(color='#AAAAAA', width=0.5),
    hoverinfo='none',
    showlegend=False,
    opacity=0.25,
)]

# Step 4: node traces — one per type
for t in sorted(_grp.keys()):
    nodes = _grp[t]
    sz = max(2, min(5, 100 // max(len(nodes), 1) + 2))
    traces.append(go.Scatter3d(
        x=[node_xyz[nd][0] for nd in nodes],
        y=[node_xyz[nd][1] for nd in nodes],
        z=[node_xyz[nd][2] for nd in nodes],
        mode='markers',
        marker=dict(size=sz, color=_NC.get(t, '#DDD'),
                    opacity=0.80, line=dict(width=0.2, color='#333')),
        text=[_sh(nd) for nd in nodes],
        hoverinfo='text+name',
        name=f'{t} ({len(nodes):,})',
    ))

_ax = dict(showgrid=False, zeroline=False, showticklabels=False, showbackground=False)
go.Figure(
    data=traces,
    layout=go.Layout(
        title=(f'Base KG — temporal layout  '
               f'({G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges)  '
               f'[x = time →, y = node type lane]'),
        scene=dict(xaxis=dict(**_ax, title='time →'), yaxis=_ax, zaxis=_ax),
        margin=dict(l=0, r=0, b=0, t=50), height=650,
        legend=dict(font=dict(size=9), itemsizing='constant'),
    )
).show()


In [ ]:
import time as _time

growth_snapshots = []
_SNAPSHOT_EVERY  = 1000   # heavier metrics computed at these intervals

for i, row in df.iterrows():

    diff       = diff_map.get(row.commit_id)
    true_label = int(row.buggy)

    # ── STEP 1: Extract KG features ── G knows nothing about this commit yet ──
    _t0_ext = _time.perf_counter()
    kg       = extract_kg_features(G, row, diff)
    kg_extract_ms = (_time.perf_counter() - _t0_ext) * 1_000

    hc = [float(row[c]) for c in HC_COLS]
    x  = np.array(hc + [kg[k] for k in KG_COLS], dtype=float)

    # ── STEP 2: Predict ───────────────────────────────────────────────────────
    predicted = None
    if i >= WARMUP:
        x_sc      = scaler.transform(x.reshape(1, -1))
        predicted = int(model.predict(x_sc)[0])

    # ── STEP 3: Record result ─────────────────────────────────────────────────
    results.append({
        "commit_id":     row.commit_id,
        "author_date":   row.author_date,
        "true_label":    true_label,
        "predicted":     predicted,
        "kg_extract_ms": kg_extract_ms,
        "kg_update_ms":  None,
        "n_nodes":       None,
        "n_edges":       None,
    })

    # ── STEP 4: Update KG ─────────────────────────────────────────────────────
    prev_cid = prev_cid_map.get(row.project)
    _t0_upd  = _time.perf_counter()
    update_kg(G, row, diff, prev_cid=prev_cid)
    results[-1]["kg_update_ms"] = (_time.perf_counter() - _t0_upd) * 1_000
    results[-1]["n_nodes"]      = G.number_of_nodes()
    results[-1]["n_edges"]      = G.number_of_edges()
    prev_cid_map[row.project]   = f"commit:{row.commit_id}"

    # ── Online model update ───────────────────────────────────────────────────
    scaler.partial_fit(x.reshape(1, -1))
    x_sc = scaler.transform(x.reshape(1, -1))
    model.partial_fit(x_sc, [true_label], classes=[0, 1])

    # ── Growth snapshot (heavier metrics, every _SNAPSHOT_EVERY commits) ──────
    if (i + 1) % _SNAPSHOT_EVERY == 0 or i == len(df) - 1:
        _n  = G.number_of_nodes()
        _e  = G.number_of_edges()
        _tc = Counter(nx.get_node_attributes(G, 'type').values())
        growth_snapshots.append({
            'commit_idx':     i + 1,
            'n_nodes':        _n,
            'n_edges':        _e,
            'avg_out_degree': _e / _n if _n else 0.0,
            'density':        nx.density(G),
            'n_weakly_cc':    nx.number_weakly_connected_components(G),
            'size_mb_est':    round(_n * 0.0005 + _e * 0.00015, 1),
            **{f'type_{t}': c for t, c in _tc.items()},
        })
        print(f"  {i+1:,} / {len(df):,}  [snapshot]  "
              f"{_n:,} nodes · {_e:,} edges · "
              f"{growth_snapshots[-1]['n_weakly_cc']:,} WCCs")
    elif (i + 1) % 1000 == 0:
        print(f"  {i+1:,} / {len(df):,} commits processed …")

print(f"\nDone. G has {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

In [ ]:
import matplotlib.pyplot as plt

timing_df = pd.DataFrame(results)[["commit_id", "author_date", "kg_extract_ms", "kg_update_ms"]]

_ROLL = 100   # rolling window (commits) for smoothing

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    timing_df.index,
    timing_df["kg_update_ms"].rolling(_ROLL, min_periods=1).mean(),
    label=f"KG update  (rolling mean {_ROLL})",
    color="#4C72B0", linewidth=1.4,
)
ax.plot(
    timing_df.index,
    timing_df["kg_extract_ms"].rolling(_ROLL, min_periods=1).mean(),
    label=f"KG feature extraction  (rolling mean {_ROLL})",
    color="#C44E52", linewidth=1.4,
)

ax.fill_between(
    timing_df.index,
    timing_df["kg_update_ms"].rolling(_ROLL, min_periods=1).quantile(0.25),
    timing_df["kg_update_ms"].rolling(_ROLL, min_periods=1).quantile(0.75),
    alpha=0.15, color="#4C72B0",
)
ax.fill_between(
    timing_df.index,
    timing_df["kg_extract_ms"].rolling(_ROLL, min_periods=1).quantile(0.25),
    timing_df["kg_extract_ms"].rolling(_ROLL, min_periods=1).quantile(0.75),
    alpha=0.15, color="#C44E52",
)

ax.set_xlabel("Commit index (chronological order)", fontsize=11)
ax.set_ylabel("Time (ms)", fontsize=11)
ax.set_title("Per-commit KG operation time over the full commit stream", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nUpdate   — mean: {timing_df.kg_update_ms.mean():.2f} ms   "
      f"median: {timing_df.kg_update_ms.median():.2f} ms   "
      f"p95: {timing_df.kg_update_ms.quantile(0.95):.2f} ms")
print(f"Extraction— mean: {timing_df.kg_extract_ms.mean():.2f} ms   "
      f"median: {timing_df.kg_extract_ms.median():.2f} ms   "
      f"p95: {timing_df.kg_extract_ms.quantile(0.95):.2f} ms")

In [ ]:
_growth_df = pd.DataFrame(results)[['n_nodes', 'n_edges']].copy()
_snap_df   = pd.DataFrame(growth_snapshots) if growth_snapshots else pd.DataFrame()

_NODE_COLORS_GROWTH = {
    'COMMIT': '#4C72B0', 'TIME': '#55A868', 'INTERVAL': '#FF8C00',
    'LABEL': '#C44E52', 'AUTHOR': '#8172B2', 'FILE': '#937860',
    'FILE_TYPE': '#DA8BC3', 'DIR': '#8C8C8C', 'EXTERNAL_PACKAGE': '#CCB974',
    'CLASS': '#64B5CD', 'FUNCTION': '#1F77B4', 'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE': '#98DF8A', 'DATA_TYPE': '#FF9896', 'ISSUE': '#FFBB78',
    'BRANCH': '#17BECF',
}

_fmt_k = plt.FuncFormatter(lambda x, _: f'{int(x/1000)}K' if x >= 1000 else str(int(x)))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('KG Growth Metrics over the Commit Stream', fontsize=14, fontweight='bold')

# ── 1. Total node count ───────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(_growth_df.index, _growth_df['n_nodes'], color='#4C72B0', linewidth=0.8)
ax.set_title('Total Node Count')
ax.set_xlabel('Commit index'); ax.set_ylabel('Nodes')
ax.yaxis.set_major_formatter(_fmt_k); ax.grid(True, alpha=0.3)

# ── 2. Total edge count ───────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(_growth_df.index, _growth_df['n_edges'], color='#C44E52', linewidth=0.8)
ax.set_title('Total Edge Count')
ax.set_xlabel('Commit index'); ax.set_ylabel('Edges')
ax.yaxis.set_major_formatter(_fmt_k); ax.grid(True, alpha=0.3)

# ── 3. Average out-degree ─────────────────────────────────────────────
ax = axes[0, 2]
_avg_deg = _growth_df['n_edges'] / _growth_df['n_nodes'].replace(0, float('nan'))
ax.plot(_growth_df.index, _avg_deg, color='#55A868', linewidth=0.8)
ax.set_title('Average Out-Degree  (edges / nodes)')
ax.set_xlabel('Commit index'); ax.set_ylabel('avg out-degree')
ax.grid(True, alpha=0.3)

# ── 4. Estimated size on disk ─────────────────────────────────────────
# Formula: ≈ 500 B / node  +  150 B / edge  (GraphML serialization estimate)
ax = axes[1, 0]
_size_mb = _growth_df['n_nodes'] * 0.0005 + _growth_df['n_edges'] * 0.00015
ax.plot(_growth_df.index, _size_mb, color='#8172B2', linewidth=0.8)
ax.set_title('Est. Size on Disk  (500 B/node + 150 B/edge)')
ax.set_xlabel('Commit index'); ax.set_ylabel('MB (estimate)')
ax.grid(True, alpha=0.3)

# ── 5. Graph density ──────────────────────────────────────────────────
ax = axes[1, 1]
if not _snap_df.empty:
    ax.plot(_snap_df['commit_idx'], _snap_df['density'], 'o-',
            color='#CCB974', linewidth=1.2, markersize=5)
ax.set_title('Graph Density  (snapshots every 1000 commits)')
ax.set_xlabel('Commit index'); ax.set_ylabel('density')
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.grid(True, alpha=0.3)

# ── 6. Weakly connected components ───────────────────────────────────
ax = axes[1, 2]
if not _snap_df.empty:
    ax.plot(_snap_df['commit_idx'], _snap_df['n_weakly_cc'], 's-',
            color='#937860', linewidth=1.2, markersize=5)
ax.set_title('Weakly Connected Components  (snapshots)')
ax.set_xlabel('Commit index'); ax.set_ylabel('# WCCs')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Stacked area: node type breakdown over time ───────────────────────
if not _snap_df.empty:
    _tcols  = sorted(c for c in _snap_df.columns if c.startswith('type_'))
    _tnames = [c[5:] for c in _tcols]
    _tvals  = _snap_df[_tcols].fillna(0).values.T
    _tcolors = [_NODE_COLORS_GROWTH.get(t, '#AAAAAA') for t in _tnames]

    fig2, ax2 = plt.subplots(figsize=(14, 5))
    ax2.stackplot(_snap_df['commit_idx'], _tvals,
                  labels=_tnames, colors=_tcolors, alpha=0.85)
    ax2.set_title('Node Type Breakdown over Commit Stream  (stacked area)', fontsize=13)
    ax2.set_xlabel('Commit index'); ax2.set_ylabel('Node count')
    ax2.yaxis.set_major_formatter(_fmt_k)
    ax2.legend(loc='upper left', fontsize=8, ncol=3, framealpha=0.85)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ── Final summary ─────────────────────────────────────────────────────
if not _snap_df.empty:
    _last = _snap_df.iloc[-1]
    print(f"Final state ({int(_last['commit_idx']):,} commits processed):")
    print(f"  Nodes              : {int(_last['n_nodes']):,}")
    print(f"  Edges              : {int(_last['n_edges']):,}")
    print(f"  Avg out-degree     : {_last['avg_out_degree']:.2f}")
    print(f"  Density            : {_last['density']:.2e}")
    print(f"  Weakly-CC          : {int(_last['n_weakly_cc']):,}")
    print(f"  Est. disk size     : {_last['size_mb_est']:.1f} MB")

## 6 — Evaluation Metrics

After the online loop completes, measure the model's performance using standard classification metrics.
Evaluation begins after the WARMUP period (commit index 50) to allow the model and scaler to stabilize.
Reports **precision, recall, F1-score** per class (clean vs. buggy) and a **confusion matrix**.

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["predicted"])
y_true = results_df.true_label.astype(int)
y_pred = results_df.predicted.astype(int)

print(f"Evaluated on {len(results_df):,} commits (after {WARMUP}-commit warm-up)\n")
print(classification_report(y_true, y_pred, target_names=["clean", "buggy"]))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))

## 7 — Final Graph Inspection & Export

Inspects the **final KG** built after all 8,059 commits:
- **Node counts by type**: How many COMMIT, AUTHOR, FILE, CLASS, FUNCTION, etc. entities are present
- **Edge relation counts**: How many edges of each type (contains, parent, child, duration, etc.)
- **Sample query**: Neighborhood of the last commit, showing all its outgoing edges
- **Export**: Writes the graph to GraphML, JSON, and CSV formats for downstream use (Gephi, analysis pipelines)

GraphML sanitization step removes the reserved "id" attribute and flattens sets/lists to comma-separated strings so the output is serializable.

In [ ]:
# Node / edge type breakdown
node_counts = pd.Series(nx.get_node_attributes(G, "type")).value_counts()
edge_counts = pd.Series([d.get("rel") for _, _, d in G.edges(data=True)]).value_counts()

print(f"Nodes : {G.number_of_nodes():,}   Edges : {G.number_of_edges():,}\n")
print("── Node types ──────────────")
print(node_counts.to_string())
print("\n── Edge relation types ─────")
print(edge_counts.head(20).to_string())

In [ ]:
# Sample query: all outgoing edges from the last commit
last_cid = f"commit:{df.commit_id.iloc[-1]}"
print(f"Neighbourhood of {last_cid}:\n")
for _, nb, d in G.out_edges(last_cid, data=True):
    print(f"  --[{d['rel']}]--> {nb}  ({G.nodes[nb].get('type', '?')})")

In [ ]:
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanitize node attributes for GraphML / JSON:
# - GraphML reserves the key "id" for the node identifier; drop any "id"
#   attribute (the node's name already encodes it, e.g. "issue:GROOVY-1")
# - GraphML can't serialize Python sets - flatten to comma-separated str
# - None becomes "" so writers don't choke
for n, d in G.nodes(data=True):
    d.pop("id", None)
    for k, v in list(d.items()):
        if v is None:
            d[k] = ""
        elif isinstance(v, (set, list, tuple)):
            d[k] = ",".join(sorted(str(x) for x in v))

graphml_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.graphml"
nx.write_graphml(G, graphml_path)
print(f"GraphML : {graphml_path}")

json_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(nx.node_link_data(G), f, ensure_ascii=False, indent=2)
print(f"JSON    : {json_path}")

results_path = OUTPUT_DIR / f"{PROJECT_NAME}_online_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Results : {results_path}")

## 8 — Step-by-Step KG Growth Inspector

Independent of the main online loop, this section builds a **fresh graph** from scratch and visualizes it at key milestones.
The goal is to **inspect the structure** of the KG as it grows—seeing what entities emerge and how they interconnect.

Checkpoints are at 1, 2, 3, 4, 5, 6, 26, 60, 200, and 500 commits.
Each checkpoint produces:
- **2D visualization** (matplotlib spring layout): useful for counting nodes and spotting connected components
- **3D visualization** (plotly): interactive, rotatable view to explore structure in 3D space
- **Node counts by type**: quantifies growth of each entity tier

Node colors are consistent across all plots: COMMIT (blue), AUTHOR (purple), FILE (brown), FUNCTION (dark blue), etc.
Early checkpoints (0–6) show sparse structure; later ones (200, 500) reveal the rich interconnectedness of the KG.

## 8.0 — Visualization Helpers

Utility functions for consistent graph rendering across all checkpoints.

**draw_kg_2d(G, title)**:
- Uses NetworkX spring layout with seed=42 for reproducibility
- Node size adapts based on graph size (smaller for dense graphs to avoid clutter)
- Spring constant `k` adjusted to prevent overlaps
- Labels shown only if the graph has ≤40 nodes (readability threshold)
- Legend displays node type and count

**draw_kg_3d(G, title)**:
- Uses 3D spring layout for deeper structure exploration
- Edges rendered as lines connecting 3D coordinates
- Nodes as interactive markers; hover shows shortened node ID
- Runs in Plotly for interactivity (rotate, zoom, pan)

**Helper functions**:
- `_groups(G)`: Partition nodes by type
- `_short(nid, mx=18)`: Truncate node IDs for display readability
- `_stats(G)`: Print concise summary (node/edge count, type breakdown)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go

NODE_COLORS = {
    'COMMIT':             '#4C72B0',
    'TIME':               '#55A868',
    'INTERVAL':           '#FF8C00',
    'LABEL':              '#C44E52',
    'AUTHOR':             '#8172B2',
    'FILE':               '#937860',
    'FILE_TYPE':          '#DA8BC3',
    'DIR':                '#8C8C8C',
    'EXTERNAL_PACKAGE':   '#CCB974',
    'CLASS':              '#64B5CD',
    'FUNCTION':           '#1F77B4',
    'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE':           '#98DF8A',
    'DATA_TYPE':          '#FF9896',
    'ISSUE':              '#FFBB78',
    'BRANCH':             '#17BECF',
}

def _groups(G):
    g = defaultdict(list)
    for n, d in G.nodes(data=True):
        g[d.get('type', 'UNKNOWN')].append(n)
    return g

def _short(nid, mx=18):
    s = nid.split(':', 1)[-1]
    return s[:mx] + '...' if len(s) > mx else s

def _stats(G):
    g = _groups(G)
    print(f'  nodes={G.number_of_nodes()}  edges={G.number_of_edges()}')
    print('  ' + '  '.join(f'{t}={len(ns)}' for t, ns in sorted(g.items())))


def draw_kg_2d(G, title=''):
    if G.number_of_nodes() == 0:
        print(f'[2D] {title}: empty graph.'); return
    n   = G.number_of_nodes()
    pos = nx.spring_layout(G, seed=42, k=3.0/(n**0.5) if n>1 else 2.0, iterations=60)
    grp = _groups(G)
    fig, ax = plt.subplots(figsize=(14, 8))
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2, arrows=True,
                           edge_color='#999', width=0.6,
                           connectionstyle='arc3,rad=0.08')
    sz = max(20, 400 - n*4)
    for t, ns in grp.items():
        nx.draw_networkx_nodes(G, pos, nodelist=ns, ax=ax,
                               node_color=NODE_COLORS.get(t, '#DDD'),
                               node_size=sz, alpha=0.9)
    if n <= 40:
        nx.draw_networkx_labels(G, pos,
                                labels={nd: _short(nd) for nd in G.nodes()},
                                ax=ax, font_size=6)
    patches = [mpatches.Patch(color=NODE_COLORS.get(t, '#DDD'),
                              label=f'{t} ({len(ns)})')
               for t, ns in sorted(grp.items())]
    ax.legend(handles=patches, loc='upper left', fontsize=7, framealpha=0.85, ncol=2)
    ax.set_title(f'{title}  |  nodes={n}  edges={G.number_of_edges()}', fontsize=11)
    ax.axis('off'); plt.tight_layout(); plt.show()


def draw_kg_3d(G, title=''):
    if G.number_of_nodes() == 0:
        print(f'[3D] {title}: empty graph.'); return
    n   = G.number_of_nodes()
    pos = nx.spring_layout(G, dim=3, seed=42, k=2.0/(n**0.5), iterations=60)
    ex, ey, ez = [], [], []
    for u, v in G.edges():
        x0,y0,z0=pos[u]; x1,y1,z1=pos[v]
        ex+=[x0,x1,None]; ey+=[y0,y1,None]; ez+=[z0,z1,None]
    traces = [go.Scatter3d(x=ex,y=ey,z=ez,mode='lines',
                           line=dict(color='#AAA',width=1),
                           hoverinfo='none',showlegend=False)]
    sz = max(3, 10 - n//20)
    for t, ns in sorted(_groups(G).items()):
        traces.append(go.Scatter3d(
            x=[pos[nd][0] for nd in ns],
            y=[pos[nd][1] for nd in ns],
            z=[pos[nd][2] for nd in ns],
            mode='markers',
            marker=dict(size=sz,color=NODE_COLORS.get(t,'#DDD'),
                        opacity=0.88,line=dict(width=0.5,color='#333')),
            text=[_short(nd,30) for nd in ns],
            hoverinfo='text+name',
            name=f'{t} ({len(ns)})',
        ))
    ax_s = dict(showgrid=False,zeroline=False,showticklabels=False,showbackground=False)
    go.Figure(
        data=traces,
        layout=go.Layout(
            title=f'{title}  |  nodes={n}  edges={G.number_of_edges()}',
            scene=dict(xaxis=ax_s,yaxis=ax_s,zaxis=ax_s),
            margin=dict(l=0,r=0,b=0,t=45),height=550,
            legend=dict(font=dict(size=9),itemsizing='constant'),
        )
    ).show()


print('draw_kg_2d / draw_kg_3d ready.')

### 8.1 — Commits 0 → 6, One at a Time

Add commits incrementally and visualize after each one.
This is the densest inspection phase: each cell shows what a single commit contributes.
By step 6, the graph has begun to populate with AUTHOR, FILE, DIR, FUNCTION, VARIABLE, and CLASS entities.

In [ ]:
G_insp    = nx.MultiDiGraph()
prev_insp = None

for step in range(1, 7):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    result    = timed_update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'
    print_commit_log(row, step, result, total=len(df))
    draw_kg_2d(G_insp, title=f'After commit {step}')
    draw_kg_3d(G_insp, title=f'After commit {step}')

### 8.2 — Checkpoint: 26 Commits

Add 20 more commits (commits 7–26). The graph begins to show interconnectedness—multiple FILE nodes, recurring AUTHOR, and denser FUNCTION networks.

In [ ]:
_batch_results = []
for step in range(7, 27):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    result    = timed_update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'
    _batch_results.append(result)
    print_commit_log(row, step, result, total=len(df))

print_batch_summary('commits 7 – 26', _batch_results, df.iloc[6:26])
draw_kg_2d(G_insp, title='26 commits')
draw_kg_3d(G_insp, title='26 commits')

### 8.3 — Checkpoint: 60 Commits

Add 34 more commits (commits 27–60). The graph is now moderately complex with clear author communities and file hierarchies.

In [ ]:
_batch_results = []
for step in range(27, 61):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    result    = timed_update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'
    _batch_results.append(result)
    print_commit_log(row, step, result, total=len(df))

print_batch_summary('commits 27 – 60', _batch_results, df.iloc[26:60])
draw_kg_2d(G_insp, title='60 commits')
draw_kg_3d(G_insp, title='60 commits')

### 8.4 — Checkpoint: 200 Commits

Add 34 more commits (commits 61-200). The graph is now moderately complex with clear author communities and file hierarchies.

In [ ]:
_batch_results = []
for step in range(61, 201):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    result    = timed_update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'
    _batch_results.append(result)
    print_commit_log(row, step, result, total=len(df))

print_batch_summary('commits 61 – 200', _batch_results, df.iloc[60:200])
draw_kg_2d(G_insp, title='200 commits')
draw_kg_3d(G_insp, title='200 commits')

In [ ]:
for step in range(61, 201):
    row = df.iloc[step - 1]
    update_kg(G_insp, row, diff_map.get(row.commit_id), prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'

print('=' * 55)
print('CHECKPOINT: 200 commits  (+140 from previous)')
print('=' * 55)
_stats(G_insp)
draw_kg_2d(G_insp, title='200 commits')
draw_kg_3d(G_insp, title='200 commits')

### 8.5 — Checkpoint: 500 Commits

Add 34 more commits (commits 201-500). The graph is now moderately complex with clear author communities and file hierarchies.

In [ ]:
_batch_results = []
for step in range(201, 501):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    result    = timed_update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'
    _batch_results.append(result)
    print_commit_log(row, step, result, total=len(df))

print_batch_summary('commits 201 – 500', _batch_results, df.iloc[200:500])
draw_kg_2d(G_insp, title='500 commits')
draw_kg_3d(G_insp, title='500 commits')

In [ ]:
for step in range(201, 501):
    row = df.iloc[step - 1]
    update_kg(G_insp, row, diff_map.get(row.commit_id), prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'

print('=' * 55)
print('CHECKPOINT: 500 commits  (+300 from previous)')
print('=' * 55)
_stats(G_insp)
draw_kg_2d(G_insp, title='500 commits')
draw_kg_3d(G_insp, title='500 commits')